# YOLO26n 청구기호 라벨 검출기 — 자동 라벨 2,071박스 학습

**데이터:** 파이프라인이 검증한 라벨 스트립 박스 32장(800·600·700번대) — 사람 라벨링 0회.
**증강:** perspective·degrees를 키워 **각도 강건성** 확보 (강한 각도 프레임은 라벨이 없어 학습셋에서 제외했으므로 증강으로 대체).

**업로드:** `yolo_labelset.zip` · GPU(T4) 런타임

In [ ]:
# 1) 설치 + 데이터 업로드
!pip install -q ultralytics
from google.colab import files
up = files.upload()   # yolo_labelset.zip
!unzip -oq yolo_labelset.zip
!ls yolo_labelset/images/train | wc -l; ls yolo_labelset/images/val | wc -l

In [ ]:
# 2) 학습 (T4 ~30-40분) — yolo26n이 없으면 yolo11n 폴백 (PRD 확정 전략)
from ultralytics import YOLO
try:
    model = YOLO('yolo26n.pt')
except Exception as e:
    print('yolo26n 불가 → yolo11n 폴백:', e)
    model = YOLO('yolo11n.pt')
model.train(data='yolo_labelset/data.yaml', epochs=120, imgsz=1280, batch=8,
            degrees=12, perspective=0.0008, shear=4, fliplr=0.0,
            hsv_v=0.5, hsv_s=0.4, mosaic=0.6, patience=30, name='call_label')

In [ ]:
# 3) 검증 수치 + 시각 확인
m = YOLO('runs/detect/call_label/weights/best.pt')
r = m.val(data='yolo_labelset/data.yaml', imgsz=1280)
print('mAP50:', r.box.map50, 'mAP50-95:', r.box.map)
import glob
res = m.predict(glob.glob('yolo_labelset/images/val/*.jpg')[0], imgsz=1280, conf=0.3, save=True)
print('예측 저장:', res[0].save_dir)

In [ ]:
# 4) ONNX 내보내기(온디바이스용) + 다운로드
m.export(format='onnx', imgsz=1280, half=False, dynamic=False)
!zip -q -j call_label_yolo.zip runs/detect/call_label/weights/best.pt runs/detect/call_label/weights/best.onnx
from google.colab import files
files.download('call_label_yolo.zip')